<a href="https://colab.research.google.com/github/ChaimElchik/GPS-Demo/blob/main/GPS_DEM_DepthAnythingDistanceSamplingVideoSequenceMediumV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Object Detection and GPS Localization Video Sequence Light


---
## Setup Environment

In [1]:
# Install necessary packages
!pip install piexif geopy pyproj torch torchvision transformers timm accelerate -q
!pip install ultralytics==8.3.18 --upgrade --quiet
!apt-get install -y exiftool -qq

# Standard library imports
import csv
import io
import json
import math
import os
import re
import subprocess
import time
import traceback
from datetime import datetime, timedelta
from pathlib import Path

# Third-party library imports
import cv2
import numpy as np
import pandas as pd
import requests
import torch
from geopy.distance import geodesic
from matplotlib import pyplot as plt
from PIL import Image
from pyproj import Transformer
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from ultralytics import YOLO

# Google Colab / IPython specific imports
from google.colab import files
from IPython.display import Image as IPImage, display

# Create output directories
os.makedirs("Detections", exist_ok=True)
os.makedirs("Processed_Output", exist_ok=True)

print("\nSetup Complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.6/876.6 kB 7.3 MB/s eta 0:00:00
Selecting previously unselected package libarchive-zip-perl.
(Reading database ... 126281 files and directories currently i


##  1. Upload Model, SRT File, Video File and Tracker Config File

In [2]:
print("--- Step 1: Upload Files ---")
print("Please upload your video, SRT file, your .pt model, and your tracker config (e.g., 'botsort.yaml').")

uploaded = files.upload()

video_path = None
srt_path = None
model_path = None
tracker_config_path = None

for fn in uploaded.keys():
    if fn.lower().endswith('.pt'):
        model_path = fn
        print(f"✅ Model file '{fn}' found.")
    elif fn.lower().endswith('.yaml'):
        tracker_config_path = fn
        print(f"✅ Tracker config file '{fn}' found.")
    elif fn.lower().endswith(('.mp4', '.mov', '.avi')):
        video_path = fn
        print(f"✅ Video file '{fn}' found.")
    elif fn.lower().endswith('.srt'):
        srt_path = fn
        print(f"✅ SRT file '{fn}' found.")

print("\n--- Verifying files ---")
if not video_path: print("❌ ERROR: Video file not uploaded.")
if not srt_path: print("❌ ERROR: SRT file not uploaded.")
if not model_path: print(f"❌ ERROR: Model .pt file not uploaded.")
if not tracker_config_path: print("❌ ERROR: Tracker config .yaml file not uploaded.")

if video_path and srt_path and model_path and tracker_config_path:
    MODEL_PATH = model_path
    print("\n--- All files ready for processing! ---")

--- Step 1: Upload Files ---
Please upload your video, SRT file, your .pt model, and your tracker config (e.g., 'botsort.yaml').


Saving Video_X_Axis.MP4 to Video_X_Axis.MP4
Saving botsortV5.yaml to botsortV5.yaml
Saving best.pt to best.pt
Saving DJI_20250618120033_0001_D.SRT to DJI_20250618120033_0001_D.SRT
✅ Video file 'Video_X_Axis.MP4' found.
✅ Tracker config file 'botsortV5.yaml' found.
✅ Model file 'best.pt' found.
✅ SRT file 'DJI_20250618120033_0001_D.SRT' found.

--- Verifying files ---

--- All files ready for processing! ---


---
## 2. Core Logic and Helper Functions


In [19]:
# --- Import Libraries ---
import os
import cv2
import numpy as np
import json
import re
import math
import time
import traceback
import csv
from pathlib import Path
import requests
import torch
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from PIL import Image

# --- Dependencies Check ---
try:
    from ultralytics import YOLO
    from geopy.distance import geodesic
    from geopy.point import Point
except ImportError as e:
    print(f"ERROR: Missing dependency - {e}. Please install required libraries.")
    print("Run: pip install ultralytics opencv-python pyproj geopy requests torch torchvision transformers timm accelerate Pillow")
    exit()

# --- Global Configuration ---
OUTPUT_DIR = "Video_Processing_Output_Medium"
# MODEL_PATH is set dynamically in the previous cell.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEPTH_MODEL_NAME = 'depth-anything/Depth-Anything-V2-Large-hf'

# --- SENSOR CONFIGURATION ---
SENSOR_WIDTH_MM = 17.3
SENSOR_HEIGHT_MM = 13.0
print(f"INFO: Using Sensor Size {SENSOR_WIDTH_MM}mm x {SENSOR_HEIGHT_MM} (Mavic 3 Pro Main Cam).")
print(f"INFO: Using device: {DEVICE} for deep learning models.")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- SRT Parsing & Geolocation Functions ---
def parse_srt_file(srt_path):
    print(f"INFO: Parsing SRT file: {srt_path}")
    metadata_map = {}
    with open(srt_path, 'r') as f: content = f.read()
    pattern = re.compile(
        r"FrameCnt: (\d+).*?\[focal_len: ([\d\.]+)\]"
        r".*?\[latitude: ([\d\.\-]+)\] \[longitude: ([\d\.\-]+)\] "
        r"\[rel_alt: ([\d\.\-]+) abs_alt: ([\d\.\-]+)\] "
        r"\[gb_yaw: ([\d\.\-]+) gb_pitch: ([\d\.\-]+) gb_roll: ([\d\.\-]+)\]", re.DOTALL)
    for match in pattern.finditer(content):
        frame_cnt = int(match.group(1))
        metadata_map[frame_cnt] = {
            'focal_len': float(match.group(2)), 'latitude': float(match.group(3)),
            'longitude': float(match.group(4)), 'rel_alt': float(match.group(5)),
            'abs_alt': float(match.group(6)), 'gb_yaw': float(match.group(7)),
            'gb_pitch': float(match.group(8)), 'gb_roll': float(match.group(9)),}
    print(f"✅ Successfully parsed metadata for {len(metadata_map)} frames from SRT.")
    return metadata_map

def get_depth_map(frame, model, processor):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.nn.functional.interpolate(
        outputs.predicted_depth.unsqueeze(1), size=image.size[::-1], mode="bicubic", align_corners=False)
    return prediction.squeeze().cpu().numpy()

def get_dem_elevation_from_api(latitude, longitude):
    try:
        url = f"https://api.opentopodata.org/v1/eudem25m?locations={latitude},{longitude}"
        response = requests.get(url, verify=False, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['results'] and data['results'][0]['elevation'] is not None:
                return data['results'][0]['elevation']
    except requests.exceptions.RequestException: return None
    return None

def get_camera_intrinsics(f_mm, s_w_mm, s_h_mm, i_w, i_h):
    fx = i_w * f_mm / s_w_mm; fy = i_h * f_mm / s_h_mm
    cx, cy = i_w / 2, i_h / 2
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

def get_rotation_matrix(pitch_deg, yaw_deg, roll_deg):
    yaw, pitch, roll = map(math.radians, [yaw_deg, pitch_deg, roll_deg])
    Rz = np.array([[math.cos(yaw), -math.sin(yaw), 0], [math.sin(yaw), math.cos(yaw), 0], [0, 0, 1]])
    Ry = np.array([[math.cos(pitch), 0, math.sin(pitch)], [0, 1, 0], [-math.sin(pitch), 0, math.cos(pitch)]])
    Rx = np.array([[1, 0, 0], [0, math.cos(roll), -math.sin(roll)], [0, math.sin(roll), math.cos(roll)]])
    R_gimbal = Rz @ Ry @ Rx
    R_cam_to_body = np.array([[0, 1, 0], [0, 0, 1], [1, 0, 0]]).T
    return R_gimbal @ R_cam_to_body

def calculate_destination_gps(origin_lat, origin_lon, east_m, north_m):
    bearing = math.degrees(math.atan2(east_m, north_m))
    distance_meters = math.hypot(east_m, north_m)
    destination = geodesic(meters=distance_meters).destination(Point(origin_lat, origin_lon), bearing)
    return destination.latitude, destination.longitude

def image_point_to_gps_from_depth(u, v, K, R, o_lat, o_lon, abs_depth_map):
    v_idx, u_idx = int(round(v)), int(round(u))
    if not (0 <= v_idx < abs_depth_map.shape[0] and 0 <= u_idx < abs_depth_map.shape[1]): return None, None
    distance_to_target = abs_depth_map[v_idx, u_idx]
    K_inv = np.linalg.inv(K)
    ray_cam = K_inv @ np.array([u, v, 1])
    ray_cam_unit = ray_cam / np.linalg.norm(ray_cam)
    point_in_cam_coords = ray_cam_unit * distance_to_target
    ned_offsets = R @ point_in_cam_coords
    ned_n, ned_e = ned_offsets[0], ned_offsets[1]
    return calculate_destination_gps(o_lat, o_lon, ned_e, ned_n)

# --- Drawing & Saving Functions ---
def draw_overlays(frame, tracked_objects_data):
    # Draw the transect line first
    frame_height, frame_width, _ = frame.shape
    cv2.line(frame, (frame_width // 2, 0), (frame_width // 2, frame_height), (255, 0, 0), 2)

    for data in tracked_objects_data:
        x1, y1, x2, y2 = data['box']
        obj_id = data['id']; conf = data['conf']
        cv2.rectangle(frame, (x1, y1), (x2, y2), (128, 0, 128), 2)
        text = f"ID: {obj_id} | Conf: {conf:.2f}"
        if 'gps' in data:
            lat, lon = data['gps']
            text += f" | GPS: {lat:.5f}, {lon:.5f}"
        if 'transect_dist' in data:
            # Draw line from object center to transect line
            center_u = int((x1 + x2) / 2)
            center_v = int((y1 + y2) / 2)
            cv2.line(frame, (center_u, center_v), (frame_width // 2, center_v), (0, 0, 255), 1)
            # Add transect distance to the label
            text += f" | Dist: {data['transect_dist']:.2f}m"

        cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (128, 0, 128), 2)
    return frame

def save_frame_by_frame_log(all_results):
    filepath = os.path.join(OUTPUT_DIR, 'frame_by_frame_log.csv')
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        # --- CORRECTED: Add the new columns to the header ---
        writer.writerow(['Frame', 'Timestamp', 'ObjectID', 'Latitude', 'Longitude', 'Confidence', 'center_u', 'center_v'])
        for result in all_results:
            # --- CORRECTED: Write the new data to the row ---
            writer.writerow([
                result['frame'], result['timestamp'], result['id'],
                f"{result['lat']:.6f}", f"{result['lon']:.6f}", f"{result['conf']:.4f}",
                f"{result['center_u']:.2f}", f"{result['center_v']:.2f}"
            ])
    print(f"✅ Frame-by-frame log saved to: {filepath}")

def save_unique_objects_summary(all_results):
    filepath = os.path.join(OUTPUT_DIR, 'unique_objects_first_seen.csv')
    first_seen = {}
    for result in all_results:
        obj_id = result['id']
        if obj_id not in first_seen:
            first_seen[obj_id] = {'id': obj_id, 'timestamp': result['timestamp'],
                'lat': result['lat'], 'lon': result['lon'], 'conf': result['conf']}
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['ObjectID', 'TimestampFirstSeen', 'Latitude', 'Longitude', 'Confidence'])
        for obj_id in sorted(first_seen.keys()):
            data = first_seen[obj_id]
            writer.writerow([data['id'], data['timestamp'], f"{data['lat']:.6f}",
                f"{data['lon']:.6f}", f"{data['conf']:.4f}"])
    print(f"✅ Unique objects summary saved to: {filepath}")


INFO: Using Sensor Size 17.3mm x 13.0 (Mavic 3 Pro Main Cam).
INFO: Using device: cpu for deep learning models.


---
## 3. Main Execution Block

In [20]:
def main_medium_light_pipeline(video_path, srt_path, model_path, tracker_config):
    start_time = time.time()
    print("\n--- 🚀 Starting MEDIUM-LIGHT Video Processing Pipeline (Hybrid Trigger) 🚀 ---")

    # --- User-configurable threshold ---
    # This value determines how far the drone must move (in meters) before a new depth map is calculated.
    # Lower values are more accurate but slower. Higher values are faster but less accurate.
    DRONE_MOVEMENT_THRESHOLD_METERS = 2.0

    try:
        yolo_model = YOLO(model_path)
        depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_NAME)
        depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_NAME).to(DEVICE)
        srt_metadata = parse_srt_file(srt_path)
    except Exception as e:
        print(f"❌ FATAL ERROR during initialization: {e}")
        return

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ ERROR: Cannot open video file {video_path}")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"INFO: Video Properties: {frame_width}x{frame_height} @ {fps:.2f} FPS, {total_frames} total frames.")

    output_video_path = os.path.join(OUTPUT_DIR, 'annotated_video_medium.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
    print(f"INFO: Output video will be saved to: {output_video_path}")

    cached_geolocation_data = None
    all_seen_ids = set()

    frame_count = 0
    all_frame_results = []

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_count += 1
        timestamp = frame_count / fps
        print(f"\n--- Processing Frame {frame_count}/{total_frames} (Timestamp: {timestamp:.2f}s) ---")

        if frame_count not in srt_metadata:
            print(f"⚠️ WARNING: No metadata in SRT for frame {frame_count}. Writing original frame.")
            out_video.write(frame)
            continue
        meta = srt_metadata[frame_count]

        results = yolo_model.track(frame, persist=True, tracker=tracker_config, conf=0.5, verbose=False)

        tracked_objects_data = []
        current_frame_ids = set()
        for result in results:
            if result.boxes.id is not None:
                boxes = result.boxes.xyxy.cpu().numpy().astype(int)
                ids = result.boxes.id.cpu().numpy().astype(int)
                confs = result.boxes.conf.cpu().numpy()
                for i in range(len(ids)):
                    tracked_objects_data.append({'id': ids[i], 'box': boxes[i], 'conf': confs[i]})
                    current_frame_ids.add(ids[i])

        if not tracked_objects_data:
            print("INFO: No objects detected in this frame.")
            cv2.line(frame, (frame_width // 2, 0), (frame_width // 2, frame_height), (255, 0, 0), 2)
            out_video.write(frame)
            continue

        print(f"INFO: Tracking {len(tracked_objects_data)} objects with IDs: {current_frame_ids}")

        try:
            recalculate_depth = False
            if cached_geolocation_data is None:
                recalculate_depth = True
                print("INFO: First detection frame. Calculating new depth map...")
            else:
                is_new_object_detected = not current_frame_ids.issubset(all_seen_ids)
                if is_new_object_detected:
                    recalculate_depth = True
                    print("INFO: New object detected. Recalculating depth map for best accuracy...")
                else:
                    last_coords = cached_geolocation_data['drone_coords']
                    current_coords = (meta['latitude'], meta['longitude'])
                    distance_moved = geodesic(last_coords, current_coords).meters
                    if distance_moved > DRONE_MOVEMENT_THRESHOLD_METERS:
                        recalculate_depth = True
                        print(f"INFO: Drone moved {distance_moved:.2f}m (>{DRONE_MOVEMENT_THRESHOLD_METERS}m). Recalculating depth map...")
                    else:
                        print(f"INFO: No new objects and drone moved {distance_moved:.2f}m. Reusing cached depth map.")

            if recalculate_depth:
                ground_elevation = get_dem_elevation_from_api(meta['latitude'], meta['longitude'])
                base_agl = (meta['abs_alt'] - ground_elevation) if ground_elevation is not None else meta['rel_alt']
                relative_depth_map = get_depth_map(frame, depth_model, depth_processor)
                scale_factor = base_agl / np.mean(relative_depth_map)

                cached_geolocation_data = {
                    'absolute_depth_map': relative_depth_map * scale_factor,
                    'K': get_camera_intrinsics(meta['focal_len'], SENSOR_WIDTH_MM, SENSOR_HEIGHT_MM, frame_width, frame_height),
                    'R': get_rotation_matrix(meta['gb_pitch'], meta['gb_yaw'], meta['gb_roll']),
                    'drone_coords': (meta['latitude'], meta['longitude']),
                    'base_agl': base_agl
                }
                all_seen_ids.update(current_frame_ids)

            K = cached_geolocation_data['K']
            R = cached_geolocation_data['R']
            drone_coords = cached_geolocation_data['drone_coords']
            absolute_depth_map = cached_geolocation_data['absolute_depth_map']
            base_agl = cached_geolocation_data['base_agl']

            gsd = (base_agl * SENSOR_WIDTH_MM) / (meta['focal_len'] * frame_width)

            for obj_data in tracked_objects_data:
                box = obj_data['box']
                center_u, center_v = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2

                pixel_distance = abs(center_u - (frame_width / 2))
                meter_distance = pixel_distance * gsd
                obj_data['transect_dist'] = meter_distance

                lat, lon = image_point_to_gps_from_depth(center_u, center_v, K, R, drone_coords[0], drone_coords[1], absolute_depth_map)
                if lat is not None and lon is not None:
                    obj_data['gps'] = (lat, lon)
                    # --- This is the corrected line that logs the pixel coordinates ---
                    all_frame_results.append({
                        'frame': frame_count, 'timestamp': f"{timestamp:.3f}", 'id': obj_data['id'],
                        'lat': lat, 'lon': lon, 'conf': obj_data['conf'],
                        'center_u': center_u, 'center_v': center_v
                    })

            annotated_frame = draw_overlays(frame.copy(), tracked_objects_data)
            out_video.write(annotated_frame)

        except Exception as e:
            print(f"❌ ERROR processing frame {frame_count}: {e}")
            traceback.print_exc()
            out_video.write(frame)
            continue

    cap.release()
    out_video.release()
    print("\n\n--- ✅ Medium-Light Video Processing Complete ---")

    if all_frame_results:
        save_frame_by_frame_log(all_frame_results)
        save_unique_objects_summary(all_frame_results)
        print(f"✅ Annotated video saved to: {output_video_path}")
    else:
        print("INFO: No objects were successfully geolocated in the video.")

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"\nTotal process completed in {elapsed_time:.2f} seconds.")
    if total_frames > 0:
        print(f"Average time per frame: {elapsed_time / total_frames:.3f} seconds.")

if 'video_path' in locals() and video_path and 'srt_path' in locals() and srt_path and 'model_path' in locals() and model_path and 'tracker_config_path' in locals() and tracker_config_path:
    main_medium_light_pipeline(video_path, srt_path, model_path, tracker_config_path)
else:
    print("\n❌ Please run Cell 1 to upload all required files before running this cell.")




--- 🚀 Starting MEDIUM-LIGHT Video Processing Pipeline (Hybrid Trigger) 🚀 ---
INFO: Parsing SRT file: DJI_20250618120033_0001_D.SRT
✅ Successfully parsed metadata for 1931 frames from SRT.
INFO: Video Properties: 1920x1080 @ 29.97 FPS, 343 total frames.
INFO: Output video will be saved to: Video_Processing_Output_Medium/annotated_video_medium.mp4

--- Processing Frame 1/343 (Timestamp: 0.03s) ---
INFO: No objects detected in this frame.

--- Processing Frame 2/343 (Timestamp: 0.07s) ---
INFO: No objects detected in this frame.

--- Processing Frame 3/343 (Timestamp: 0.10s) ---
INFO: No objects detected in this frame.

--- Processing Frame 4/343 (Timestamp: 0.13s) ---
INFO: No objects detected in this frame.

--- Processing Frame 5/343 (Timestamp: 0.17s) ---
INFO: No objects detected in this frame.

--- Processing Frame 6/343 (Timestamp: 0.20s) ---
INFO: No objects detected in this frame.

--- Processing Frame 7/343 (Timestamp: 0.23s) ---
INFO: No objects detected in this frame.

--- Pr

/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.opentopodata.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



--- Processing Frame 69/343 (Timestamp: 2.30s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.00m. Reusing cached depth map.

--- Processing Frame 70/343 (Timestamp: 2.34s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.00m. Reusing cached depth map.

--- Processing Frame 71/343 (Timestamp: 2.37s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.00m. Reusing cached depth map.

--- Processing Frame 72/343 (Timestamp: 2.40s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.00m. Reusing cached depth map.

--- Processing Frame 73/343 (Timestamp: 2.44s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.00m. Reusing cached depth map.

--- Processing Frame 74/343 (Timestamp: 2.47s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.00m. 

/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.opentopodata.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



--- Processing Frame 151/343 (Timestamp: 5.04s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.00m. Reusing cached depth map.

--- Processing Frame 152/343 (Timestamp: 5.07s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.11m. Reusing cached depth map.

--- Processing Frame 153/343 (Timestamp: 5.11s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.11m. Reusing cached depth map.

--- Processing Frame 154/343 (Timestamp: 5.14s) ---
INFO: No objects detected in this frame.

--- Processing Frame 155/343 (Timestamp: 5.17s) ---
INFO: No objects detected in this frame.

--- Processing Frame 156/343 (Timestamp: 5.21s) ---
INFO: No objects detected in this frame.

--- Processing Frame 157/343 (Timestamp: 5.24s) ---
INFO: No objects detected in this frame.

--- Processing Frame 158/343 (Timestamp: 5.27s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: 

...


--- Processing Frame 269/343 (Timestamp: 8.98s) ---
INFO: No objects detected in this frame.

--- Processing Frame 270/343 (Timestamp: 9.01s) ---
INFO: No objects detected in this frame.

--- Processing Frame 271/343 (Timestamp: 9.04s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.70m. Reusing cached depth map.

--- Processing Frame 272/343 (Timestamp: 9.08s) ---
INFO: No objects detected in this frame.

--- Processing Frame 273/343 (Timestamp: 9.11s) ---
INFO: No objects detected in this frame.

--- Processing Frame 274/343 (Timestamp: 9.14s) ---
INFO: No objects detected in this frame.

--- Processing Frame 275/343 (Timestamp: 9.18s) ---
INFO: No objects detected in this frame.

--- Processing Frame 276/343 (Timestamp: 9.21s) ---
INFO: No objects detected in this frame.

--- Processing Frame 277/343 (Timestamp: 9.24s) ---
INFO: No objects detected in this frame.

--- Processing Frame 278/343 (Timestamp: 9.28s) ---
INFO: No objects detect

/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.opentopodata.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



--- Processing Frame 280/343 (Timestamp: 9.34s) ---
INFO: No objects detected in this frame.

--- Processing Frame 281/343 (Timestamp: 9.38s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.47m. Reusing cached depth map.

--- Processing Frame 282/343 (Timestamp: 9.41s) ---
INFO: No objects detected in this frame.

--- Processing Frame 283/343 (Timestamp: 9.44s) ---
INFO: No objects detected in this frame.

--- Processing Frame 284/343 (Timestamp: 9.48s) ---
INFO: No objects detected in this frame.

--- Processing Frame 285/343 (Timestamp: 9.51s) ---
INFO: No objects detected in this frame.

--- Processing Frame 286/343 (Timestamp: 9.54s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.93m. Reusing cached depth map.

--- Processing Frame 287/343 (Timestamp: 9.58s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 1.29m. Reusing cached depth map.

--- Processin

/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.opentopodata.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



--- Processing Frame 300/343 (Timestamp: 10.01s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.00m. Reusing cached depth map.

--- Processing Frame 301/343 (Timestamp: 10.04s) ---
INFO: No objects detected in this frame.

--- Processing Frame 302/343 (Timestamp: 10.08s) ---
INFO: No objects detected in this frame.

--- Processing Frame 303/343 (Timestamp: 10.11s) ---
INFO: No objects detected in this frame.

--- Processing Frame 304/343 (Timestamp: 10.14s) ---
INFO: No objects detected in this frame.

--- Processing Frame 305/343 (Timestamp: 10.18s) ---
INFO: No objects detected in this frame.

--- Processing Frame 306/343 (Timestamp: 10.21s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.11m. Reusing cached depth map.

--- Processing Frame 307/343 (Timestamp: 10.24s) ---
INFO: Tracking 1 objects with IDs: {np.int64(1)}
INFO: No new objects and drone moved 0.11m. Reusing cached depth map.

--- P

---
## 4.  Display First Detections Log

In [21]:
csv_path = os.path.join('Video_Processing_Output_Medium', 'unique_objects_first_seen.csv')
print(f"--- Loading results from: {csv_path} ---")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("✅ Successfully loaded the summary of unique object detections:")
    display(df)
else:
    print(f"❌ ERROR: The output file was not found at '{csv_path}'.")
    print("Please ensure the main processing cell (CELL 3) completed without errors.")


--- Loading results from: Video_Processing_Output_Medium/unique_objects_first_seen.csv ---
✅ Successfully loaded the summary of unique object detections:


,ObjectID,TimestampFirstSeen,Latitude,Longitude,Confidence
0,1,2.269,52.725852,-2.746641,0.7678


In [24]:
import pandas as pd
import os
from geopy.distance import geodesic

# Define the path to the detailed log file
log_path = os.path.join('Video_Processing_Output_Medium', 'frame_by_frame_log.csv')

print(f"--- Loading detailed log from: {log_path} ---")

if os.path.exists(log_path):
    try:
        # Load the detailed log into a DataFrame
        df_log = pd.read_csv(log_path)
        print("✅ Successfully loaded the frame-by-frame log.")

        # --- Intermittent Movement Calculation (for average) ---
        print("\n--- Calculating object movement between frames... ---")

        # Sort the data by object ID and then by timestamp to ensure correct order
        df_log = df_log.sort_values(by=['ObjectID', 'Timestamp'])

        movement_records = []

        # Group by each object ID to process its path individually
        for object_id, group in df_log.groupby('ObjectID'):
            group['Prev_Latitude'] = group['Latitude'].shift(1)
            group['Prev_Longitude'] = group['Longitude'].shift(1)

            for index, row in group.iterrows():
                if pd.notna(row['Prev_Latitude']):
                    current_pos = (row['Latitude'], row['Longitude'])
                    prev_pos = (row['Prev_Latitude'], row['Prev_Longitude'])
                    distance_moved = geodesic(prev_pos, current_pos).meters
                    if distance_moved > 0.01:
                        movement_records.append({
                            'ObjectID': row['ObjectID'],
                            'DistanceMovedMeters': distance_moved
                        })

        if movement_records:
            df_movement = pd.DataFrame(movement_records)
            print("\n✅ Intermittent movement calculation complete.")

            # --- MODIFIED: Calculate Total Distance using Farthest Point ---
            print("\n--- Calculating summary statistics... ---")

            total_distance_records = []
            for object_id, group in df_log.groupby('ObjectID'):
                if len(group) > 1:
                    first_row = group.iloc[0]
                    start_pos = (first_row['Latitude'], first_row['Longitude'])

                    max_distance = 0
                    # Iterate through the rest of the points in the object's path
                    for index, row in group.iloc[1:].iterrows():
                        current_pos = (row['Latitude'], row['Longitude'])
                        # Calculate distance from the start to the current point
                        distance = geodesic(start_pos, current_pos).meters
                        # If this distance is the largest so far, update max_distance
                        if distance > max_distance:
                            max_distance = distance

                    total_distance_records.append({
                        'ObjectID': object_id,
                        'TotalDistanceMeters': max_distance
                    })

            df_total_dist = pd.DataFrame(total_distance_records)

            # 2. Calculate the average of the intermittent moves
            df_avg_move = df_movement.groupby('ObjectID')['DistanceMovedMeters'].mean().reset_index()
            df_avg_move.rename(columns={'DistanceMovedMeters': 'AverageMoveMeters'}, inplace=True)

            # 3. Merge the total distance and average move summaries
            if not df_total_dist.empty:
                df_summary = pd.merge(df_total_dist, df_avg_move, on='ObjectID', how='left')
                print("\n--- Summary of Movement per Object ---")
                display(df_summary.set_index('ObjectID'))
            else:
                 print("\n--- Summary of Movement per Object (No total distance calculated) ---")
                 display(df_avg_move.set_index('ObjectID'))

        else:
            print("\n-> No significant object movement was detected between frames.")

    except Exception as e:
        print(f"❌ An error occurred during movement calculation: {e}")
        traceback.print_exc()
else:
    print(f"❌ ERROR: The detailed log file was not found at '{log_path}'.")
    print("Please ensure the main processing cell (CELL 3) completed successfully and generated the log.")


--- Loading detailed log from: Video_Processing_Output_Medium/frame_by_frame_log.csv ---
✅ Successfully loaded the frame-by-frame log.

--- Calculating object movement between frames... ---

✅ Intermittent movement calculation complete.

--- Calculating summary statistics... ---

--- Summary of Movement per Object ---


,TotalDistanceMeters,AverageMoveMeters
ObjectID,,
1,9.783639,0.383158


In [23]:
import pandas as pd
import os
import requests
import cv2
from pathlib import Path # Import Path for the fallback

print("--- Starting Distance Sampling Analysis ---")

# --- Configuration ---
# Define paths to the files generated by the main process
log_path = os.path.join('Video_Processing_Output_Medium', 'frame_by_frame_log.csv')
video_path_for_props = video_path # Use the global video_path from Cell 1
srt_path_for_meta = srt_path # Use the global srt_path from Cell 1
output_csv_path = os.path.join('Video_Processing_Output_Medium', 'distance_sampling_data.csv')

def get_region_from_gps_once(lat, lon):
    """Performs a single reverse geocoding lookup."""
    print(f"\nINFO: Querying OpenStreetMap API for region name at {lat:.4f}, {lon:.4f}...")
    headers = {'User-Agent': 'EcologicalSurveyScript/1.0'}
    url = f"https://nominatim.openstreetmap.org/reverse?format=json&lat={lat}&lon={lon}&zoom=10"
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            data = response.json()
            address = data.get('address', {})
            name_parts = [
                address.get('nature_reserve'), address.get('national_park'),
                address.get('village'), address.get('town'), address.get('city'),
                address.get('state'), address.get('country')
            ]
            region_name = ', '.join(part for part in name_parts if part)
            if region_name:
                print(f"INFO: API lookup successful. Region: {region_name}")
                return region_name
    except requests.exceptions.RequestException as e:
        print(f"WARNING: Region API request failed. Error: {e}")
    return None

if os.path.exists(log_path):
    try:
        # Load the detailed log and SRT data
        df_log = pd.read_csv(log_path)

        # --- NEW: Verification Step ---
        required_columns = ['Latitude', 'Longitude', 'Frame', 'ObjectID', 'center_u']
        if not all(col in df_log.columns for col in required_columns):
            print(f"❌ ERROR: The log file is missing required columns. Found: {df_log.columns.to_list()}")
            print(f"Please re-run the main processing cell (Cell 3) to regenerate the log file correctly.")
        else:
            print("✅ Log file loaded successfully with all required columns.")
            srt_data = parse_srt_file(srt_path_for_meta)

            # Get video properties to find frame width/height
            cap = cv2.VideoCapture(video_path_for_props)
            frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            cap.release()

            # Get a single region label for the whole video from the first frame's coordinates
            # --- CORRECTED: Use 'Latitude' and 'Longitude' ---
            first_lat, first_lon = df_log.iloc[0]['Latitude'], df_log.iloc[0]['Longitude']
            region_label = get_region_from_gps_once(first_lat, first_lon)
            if not region_label:
                region_label = Path(video_path_for_props).stem # Fallback to video name

            # --- Main Calculation Loop ---
            distance_sampling_records = []
            for index, row in df_log.iterrows():
                # --- CORRECTED: Use 'Frame' ---
                frame_num = row['Frame']
                if frame_num in srt_data:
                    meta = srt_data[frame_num]

                    # Calculate Ground Sampling Distance (GSD) for this frame
                    gsd_w = (meta['rel_alt'] * SENSOR_WIDTH_MM) / (meta['focal_len'] * frame_width)
                    gsd_h = (meta['rel_alt'] * SENSOR_HEIGHT_MM) / (meta['focal_len'] * frame_height)

                    # --- CORRECTED: Use 'center_u' ---
                    pixel_distance = abs(row['center_u'] - (frame_width / 2))
                    meter_distance = pixel_distance * gsd_w

                    # Calculate ground coverage area and effort
                    ground_width_m = frame_width * gsd_w
                    ground_height_m = frame_height * gsd_h
                    area_sq_km = (ground_width_m * ground_height_m) / 1_000_000

                    # Append record in the desired format
                    # --- CORRECTED: Use 'ObjectID', 'Latitude', 'Longitude' ---
                    distance_sampling_records.append({
                        'Region.Label': region_label,
                        'Area.km2': f"{area_sq_km:.6f}",
                        'Sample.Label': f"frame_{frame_num}",
                        'Effort.m': f"{ground_height_m:.2f}",
                        'object': row['ObjectID'],
                        'distance': f"{meter_distance:.2f}",
                        'size': 1, # Assuming size of each detection is 1
                        'Lat': f"{row['Latitude']:.6f}",
                        'Lon': f"{row['Longitude']:.6f}"
                    })

            # --- Save the final CSV ---
            if distance_sampling_records:
                df_dist_sample = pd.DataFrame(distance_sampling_records)
                df_dist_sample.to_csv(output_csv_path, index=False)
                print(f"\n✅ Distance sampling data successfully saved to: {output_csv_path}")
                print("\n--- First 5 rows of the distance sampling data ---")
                display(df_dist_sample.head())
            else:
                print("\n-> No data to process for distance sampling.")

    except Exception as e:
        print(f"❌ An error occurred during distance sampling analysis: {e}")
        traceback.print_exc()
else:
    print(f"❌ ERROR: The detailed log file was not found at '{log_path}'.")
    print("Please ensure the main processing cell (CELL 3) completed successfully.")


--- Starting Distance Sampling Analysis ---
✅ Log file loaded successfully with all required columns.
INFO: Parsing SRT file: DJI_20250618120033_0001_D.SRT
✅ Successfully parsed metadata for 1931 frames from SRT.

INFO: Querying OpenStreetMap API for region name at 52.7259, -2.7466...
INFO: API lookup successful. Region: England, United Kingdom

✅ Distance sampling data successfully saved to: Video_Processing_Output_Medium/distance_sampling_data.csv

--- First 5 rows of the distance sampling data ---


,Region.Label,Area.km2,Sample.Label,Effort.m,object,distance,size,Lat,Lon
0,"England, United Kingdom",0.001253,frame_68.0,30.69,1.0,1.71,1,52.725852,-2.746641
1,"England, United Kingdom",0.001253,frame_69.0,30.69,1.0,1.71,1,52.725852,-2.746641
2,"England, United Kingdom",0.001253,frame_70.0,30.69,1.0,1.71,1,52.725852,-2.746641
3,"England, United Kingdom",0.001253,frame_71.0,30.69,1.0,1.71,1,52.725851,-2.746641
4,"England, United Kingdom",0.001253,frame_72.0,30.69,1.0,1.73,1,52.725851,-2.746642
